# Sales Forecasting with Prophet
### Google Colab + GitHub portfolio project

**Goal:** Forecast retail sales using historical daily data, identify trend and seasonal patterns, compare forecasts with actual sales, and generate a 30-day forward forecast.

**Dataset:** Synthetic Retail Sales (`date`, `store`, `sales`) from the public GitHub repository linked below.

**Important dataset limitation:** This CSV does not contain a promotion flag or promotion calendar. The notebook can model trend and seasonality, but it cannot estimate promotional effects without a real promotion feature. Do not invent promotion labels.

## 1. Dataset and setup

The notebook can load the CSV directly from the public source, or you can upload the file you already downloaded.

Source: [Synthetic Retail Sales CSV on GitHub](https://github.com/shibilasherinn/Retail-Sales-Forecasting/blob/main/retail_sales_synthetic.csv)

The expected columns are:
- `date`: observation date
- `store`: store identifier/name
- `sales`: daily sales value

If you prefer using your own downloaded copy, set `LOAD_FROM_URL = False` in the next cell.

In [ ]:
!pip -q install prophet seaborn scikit-learn

In [ ]:
LOAD_FROM_URL = True  # Change to False to upload your downloaded CSV

if LOAD_FROM_URL:
    url = (
        "https://raw.githubusercontent.com/"
        "shibilasherinn/Retail-Sales-Forecasting/main/"
        "retail_sales_synthetic.csv"
    )
    df = pd.read_csv(url)
else:
    from google.colab import files
    import io

    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise ValueError("Please upload the synthetic retail sales CSV file.")
    df = pd.read_csv(io.BytesIO(uploaded[csv_files[0]]))

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Normalize column names and validate the expected schema
df.columns = df.columns.str.strip().str.lower()

required_columns = {"date", "store", "sales"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        "Expected columns are date, store, and sales."
    )

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")
df["store"] = df["store"].astype("string").str.strip()

# Dates and store names are required; sales may be missing and are handled below.
df = df.dropna(subset=["date", "store"]).copy()
df = df.loc[df["store"] != ""].copy()

# If duplicate rows exist for a store/date, aggregate them to one daily total.
df = (
    df.groupby(["store", "date"], as_index=False)["sales"]
      .sum(min_count=1)
      .sort_values(["store", "date"])
)

print("Cleaned shape:", df.shape)
print("Stores:", sorted(df["store"].dropna().unique().tolist()))
print("Missing sales values:", int(df["sales"].isna().sum()))
display(df.head())

## 2. Select a store and prepare a daily time series

Choose one store to keep the example easy to understand. If the selected name does not exist, the notebook stops and shows the available names.

We create a complete daily calendar between the store's first and last dates. Missing sales are kept as missing until the chronological train/test split, so the held-out test period is not used to fill training data.

In [ ]:
available_stores = sorted(df["store"].dropna().unique().tolist())
STORE_ID = "Store_A"  # Change this to another value shown in available_stores

if STORE_ID not in available_stores:
    raise ValueError(
        f"STORE_ID={STORE_ID!r} was not found. "
        f"Available stores: {available_stores}"
    )

selected = df.loc[df["store"] == STORE_ID, ["date", "sales"]].copy()
selected = selected.groupby("date")["sales"].sum(min_count=1).sort_index()

if selected.empty:
    raise ValueError(f"No rows were found for {STORE_ID}.")

full_dates = pd.date_range(selected.index.min(), selected.index.max(), freq="D")
daily_sales = selected.reindex(full_dates)
daily_sales.index.name = "ds"
daily = daily_sales.rename("y").reset_index()

print("Selected store:", STORE_ID)
print("First date:", daily["ds"].min().date())
print("Last date:", daily["ds"].max().date())
print("Calendar days:", len(daily))
print("Missing sales after daily reindex:", int(daily["y"].isna().sum()))
display(daily.head())

## 3. Exploratory data analysis (EDA)

Explore the sales time series, weekly patterns, and monthly averages. These charts help identify potential trends and seasonality.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(daily["ds"], daily["y"], linewidth=1)
plt.title(f"Daily Sales — {STORE_ID}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

eda = daily.dropna(subset=["y"]).copy()
eda["weekday"] = eda["ds"].dt.day_name()
weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

plt.figure(figsize=(10, 4))
sns.boxplot(data=eda, x="weekday", y="y", order=weekday_order)
plt.xticks(rotation=35)
plt.title("Sales Distribution by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

eda["month"] = eda["ds"].dt.month
monthly_pattern = eda.groupby("month", as_index=False)["y"].mean()

plt.figure(figsize=(9, 4))
sns.lineplot(data=monthly_pattern, x="month", y="y", marker="o")
plt.xticks(range(1, 13))
plt.title("Average Sales by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Average daily sales")
plt.tight_layout()
plt.show()

print("Sales summary:")
display(eda["y"].describe().to_frame().T)

## 4. Chronological train/test split

We reserve the final 84 calendar days (12 weeks) as a holdout test period. All earlier dates are training data. We do not randomly shuffle time-series observations.

Missing sales in the training period are interpolated using training data only. Missing actual sales in the test period will be excluded from the error metrics.

In [ ]:
HORIZON = 84

if len(daily) <= HORIZON + 365:
    raise ValueError(
        "Not enough daily history for an 84-day test period and one year of training."
    )

train_raw = daily.iloc[:-HORIZON].copy()
test = daily.iloc[-HORIZON:].copy()

train_series = train_raw.set_index("ds")["y"].astype(float)
train_series = train_series.interpolate(method="time").ffill().bfill()

if train_series.isna().any():
    raise ValueError("Training sales still contain missing values after imputation.")

train = pd.DataFrame({"ds": train_series.index, "y": train_series.values})

print("Train range:", train["ds"].min().date(), "to", train["ds"].max().date())
print("Test range: ", test["ds"].min().date(), "to", test["ds"].max().date())
print("Training rows:", len(train))
print("Observed test sales:", int(test["y"].notna().sum()), "of", len(test))

## 5. Create a weekly seasonal-naive baseline

A forecast should be compared with a simple benchmark. This baseline uses the most recent training sales value for the same weekday (for example, the most recent training Monday for a future Monday).

In [ ]:
train_for_baseline = train.set_index("ds")["y"]
last_value_by_weekday = train_for_baseline.groupby(
    train_for_baseline.index.dayofweek
).last()

test_dates = pd.DatetimeIndex(test["ds"])
baseline_values = [
    float(last_value_by_weekday.loc[date.dayofweek])
    for date in test_dates
]

## 6. Train the Prophet model

Prophet models a changing trend and repeating weekly/yearly seasonal patterns. This dataset has no promotion field, so we do not add a promotion regressor.

For a genuine promotion analysis, add a real promotion calendar with known dates and a `promo` column, then include it as a regressor for both training dates and forecast dates.

In [ ]:
model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="multiplicative"
)

model.fit(train[["ds", "y"]])

future_test = pd.DataFrame({"ds": test_dates})
test_forecast = model.predict(future_test)

predictions = pd.DataFrame({
    "ds": test_dates,
    "actual_sales": test["y"].to_numpy(),
    "baseline_forecast": baseline_values,
    "prophet_forecast": test_forecast["yhat"].clip(lower=0).to_numpy(),
    "forecast_lower": test_forecast["yhat_lower"].clip(lower=0).to_numpy(),
    "forecast_upper": test_forecast["yhat_upper"].clip(lower=0).to_numpy()
})

display(predictions.head())

## 7. Evaluate forecasts

We report:
- **MAE:** Mean absolute error, in sales units.
- **RMSE:** Root mean squared error, which penalizes larger errors more strongly.
- **WAPE:** Total absolute error divided by total actual sales, as a percentage.

Only rows with observed actual sales are scored. Metrics are calculated on the same test dates for both models.

In [ ]:
def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual).sum()
    wape = (
        np.abs(actual - predicted).sum() / denominator * 100
        if denominator != 0 else np.nan
    )
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "WAPE (%)": wape
    }

evaluation = predictions.dropna(subset=["actual_sales"]).copy()

results = pd.DataFrame([
    {
        "Model": "Weekly seasonal-naive baseline",
        **calculate_metrics(
            evaluation["actual_sales"],
            evaluation["baseline_forecast"]
        )
    },
    {
        "Model": "Prophet",
        **calculate_metrics(
            evaluation["actual_sales"],
            evaluation["prophet_forecast"]
        )
    }
])

display(results.round(2))

## 8. Plot actual sales vs. test forecasts

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(train["ds"], train["y"], label="Training sales", alpha=0.45)
plt.plot(test["ds"], test["y"], label="Test actual sales", linewidth=2)
plt.plot(
    predictions["ds"], predictions["baseline_forecast"],
    label="Weekly baseline", linestyle="--"
)
plt.plot(
    predictions["ds"], predictions["prophet_forecast"],
    label="Prophet forecast", linewidth=2
)
plt.fill_between(
    predictions["ds"],
    predictions["forecast_lower"],
    predictions["forecast_upper"],
    alpha=0.15,
    label="Prophet uncertainty interval"
)
plt.title(f"Actual vs Forecast Sales — {STORE_ID}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

model.plot_components(test_forecast)
plt.tight_layout()
plt.show()

## 9. Generate a 30-day future forecast

After evaluating the holdout period, fit a fresh Prophet model on all historical observations. Missing values are interpolated in this full historical series, then the model forecasts the next 30 calendar days.

The future forecast does not contain promotional effects because no promotion calendar is present in this dataset.

In [ ]:
all_history = daily.set_index("ds")["y"].astype(float)
all_history = all_history.interpolate(method="time").ffill().bfill()

final_training = pd.DataFrame({
    "ds": all_history.index,
    "y": all_history.values
})

final_model = Prophet(
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode="multiplicative"
)
final_model.fit(final_training)

FUTURE_DAYS = 30
future_dates = pd.date_range(
    daily["ds"].max() + pd.Timedelta(days=1),
    periods=FUTURE_DAYS,
    freq="D"
)
future_df = pd.DataFrame({"ds": future_dates})
future_forecast = final_model.predict(future_df)

future_predictions = future_forecast[
    ["ds", "yhat", "yhat_lower", "yhat_upper"]
].copy()
future_predictions[["yhat", "yhat_lower", "yhat_upper"]] = (
    future_predictions[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)
)

display(future_predictions.head(10))

In [ ]:
plt.figure(figsize=(14, 5))
recent_history = final_training.tail(120)
plt.plot(recent_history["ds"], recent_history["y"], label="Recent actual sales")
plt.plot(
    future_predictions["ds"], future_predictions["yhat"],
    label="30-day forecast", linewidth=2
)
plt.fill_between(
    future_predictions["ds"],
    future_predictions["yhat_lower"],
    future_predictions["yhat_upper"],
    alpha=0.2,
    label="Forecast uncertainty interval"
)
plt.title(f"Next 30 Days of Forecast Sales — {STORE_ID}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Export your results

The following cell downloads the test predictions, next-30-day forecast, and model comparison metrics from Colab. These generated output files are optional project artifacts; the raw source dataset should not be committed to GitHub unless its terms allow it.

In [ ]:
predictions.to_csv("sales_forecast_test_predictions.csv", index=False)
future_predictions.to_csv("sales_forecast_next_30_days.csv", index=False)
results.to_csv("sales_forecast_model_metrics.csv", index=False)

from google.colab import files
files.download("sales_forecast_test_predictions.csv")
files.download("sales_forecast_next_30_days.csv")
files.download("sales_forecast_model_metrics.csv")

## 11. Write your findings before publishing

After running the notebook, update your GitHub README with the actual values and observations:

- Which model performed better on the 84-day holdout?
- What were the MAE, RMSE, and WAPE values?
- What weekly/yearly patterns did you observe?
- Did forecast errors cluster around particular dates or seasons?
- What could improve the forecast?

### Limitations

- The dataset is synthetic, not actual retail transaction data.
- It contains date, store, and sales only; it has no promotion flag, holiday flag, pricing, stock, or marketing variables.
- The example evaluates one store and one holdout period.
- Forecasts should be monitored and re-evaluated on new data before business use.

### Next steps

Try a rolling-origin evaluation, test other stores, compare with ARIMA/SARIMA, and add a real promotion or holiday calendar only when you have trustworthy features available at forecast time.